# Promotions and Discounts

This notebook performs comprehensive data quality checks on the Promotions_and_Discounts dataset.


## 1. Data Loading


In [1]:
import pandas as pd

# Load the Promotions_and_Discounts data
df = pd.read_csv('../datasets/Promotions_and_Discounts.csv')

print("Data loaded successfully!")
print(f"Total records: {len(df)}")


Data loaded successfully!
Total records: 50


## 2. Basic Data Exploration


In [2]:
# Display first few rows
df.head()


,Promotion ID,Product ID,Site ID,Start Date,End Date,Discount Type,Discount Amount
0,PROMO10000,PRD10033,CXUL,2024-08-29,2024-12-25,Percentage,20.00
1,PROMO10001,PRD10081,30T9,2024-05-07,2025-01-03,Percentage,21.00
2,PROMO10002,PRD10087,H75L,2024-10-13,2025-02-25,Flat,23.59
3,PROMO10003,PRD10017,FFYV,2024-07-12,2025-01-03,Flat,49.42
4,PROMO10004,PRD10018,T4UF,2024-09-09,2025-03-11,Percentage,29.00


In [3]:
# Display column names
df.columns


Index(['Promotion ID', 'Product ID', 'Site ID', 'Start Date', 'End Date',
       'Discount Type', 'Discount Amount'],
      dtype='object')

In [4]:
# Display dataset shape
df.shape


(50, 7)

In [5]:
# Display data types and basic info
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Promotion ID     50 non-null     object 
 1   Product ID       50 non-null     object 
 2   Site ID          50 non-null     object 
 3   Start Date       50 non-null     object 
 4   End Date         50 non-null     object 
 5   Discount Type    50 non-null     object 
 6   Discount Amount  50 non-null     float64
dtypes: float64(1), object(6)
memory usage: 2.9+ KB


## 3. Data Quality Checks

### 3.1 Missing Values Analysis


In [6]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("=== Missing Values Analysis ===")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


=== Missing Values Analysis ===
                 Missing Values  Percentage
Promotion ID                  0         0.0
Product ID                    0         0.0
Site ID                       0         0.0
Start Date                    0         0.0
End Date                      0         0.0
Discount Type                 0         0.0
Discount Amount               0         0.0

Total missing values: 0


### 3.2 Duplicate Records Check


In [7]:
# Check for duplicate records
duplicates_all = df.duplicated().sum()
print("=== Duplicate Records Check ===")
print(f"Total duplicate rows: {duplicates_all}")


=== Duplicate Records Check ===
Total duplicate rows: 0


### 3.3 Data Type Validation


In [8]:
# Check data types
print("=== Data Type Validation ===")
print(df.dtypes)

# Verify expected data types
expected_types = {
    'Promotion ID': 'object',
    'Product ID': 'object',
    'Site ID': 'object',
    'Start Date': 'object',
    'End Date': 'object',
    'Discount Type': 'object',
    'Discount Amount': 'float64'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns and df[col].dtype != expected:
        type_issues.append(f"{col}: expected {expected}, got {df[col].dtype}")

if type_issues:
    print("\nData Type Issues:")
    for issue in type_issues:
        print(f"  - {issue}")
else:
    print("\nAll data types are correct!")


=== Data Type Validation ===
Promotion ID        object
Product ID          object
Site ID             object
Start Date          object
End Date            object
Discount Type       object
Discount Amount    float64
dtype: object

All data types are correct!


### 3.4 Value Range Checks


In [9]:
# Check value ranges for numeric columns
print("=== Value Range Checks ===")

# Get numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"\n{col}: Min={min_val}, Max={max_val}")
    
    # Check for negative values (except where appropriate)
    if 'ID' not in col and 'Flag' not in col:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  Negative values found: {neg_count}")


=== Value Range Checks ===

Discount Amount: Min=5.0, Max=49.42


### 3.5 Categorical Value Validation


In [10]:
# Check unique values in categorical columns
print("=== Categorical Value Validation ===")

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if 'ID' not in col and 'Date' not in col and 'Name' not in col:
        unique_vals = df[col].unique()
        print(f"\n{col} unique values: {unique_vals}")


=== Categorical Value Validation ===

Discount Type unique values: ['Percentage' 'Flat']


### 3.6 Outlier Detection


In [11]:
# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===")

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check outliers for numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_columns:
    if 'ID' not in col:
        count, lower, upper = detect_outliers_iqr(col)
        print(f"\n{col}:")
        print(f"  Outliers: {count}")
        print(f"  Lower Bound: {lower:.2f}")
        print(f"  Upper Bound: {upper:.2f}")


=== Outlier Detection (IQR Method) ===

Discount Amount:
  Outliers: 0
  Lower Bound: -18.45
  Upper Bound: 66.82


## 4. Data Quality Summary


In [12]:
# Generate comprehensive data quality summary
print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   - Total Records: {len(df)}")
print(f"   - Total Columns: {len(df.columns)}")

print(f"\n2. Missing Values:")
print(f"   - Total Missing: {df.isnull().sum().sum()}")
for col in df.columns:
    missing = df[col].isnull().sum()
    print(f"   - {col}: {missing}")

print(f"\n3. Duplicate Records:")
print(f"   - Duplicate Rows: {df.duplicated().sum()}")

print(f"\n4. Data Quality Status:")
quality_passed = True

if df.isnull().sum().sum() > 0:
    print(f"   ❌ Missing values found")
    quality_passed = False
else:
    print(f"   ✓ No missing values")

if df.duplicated().sum() > 0:
    print(f"   ❌ Duplicate records found")
    quality_passed = False
else:
    print(f"   ✓ No duplicate records")

if quality_passed:
    print(f"\n✅ DATA QUALITY CHECK PASSED!")
else:
    print(f"\n❌ DATA QUALITY ISSUES DETECTED - REVIEW REQUIRED!")


DATA QUALITY SUMMARY

1. Dataset Overview:
   - Total Records: 50
   - Total Columns: 7

2. Missing Values:
   - Total Missing: 0
   - Promotion ID: 0
   - Product ID: 0
   - Site ID: 0
   - Start Date: 0
   - End Date: 0
   - Discount Type: 0
   - Discount Amount: 0

3. Duplicate Records:
   - Duplicate Rows: 0

4. Data Quality Status:
   ✓ No missing values
   ✓ No duplicate records

✅ DATA QUALITY CHECK PASSED!
